# 🔍 Fake News Detection — RoBERTa Fine-tuning (Google Colab GPU)

**Project 11 — ANLP Fake News & Misinformation Detection System**

This notebook fine-tunes `roberta-base` on the LIAR dataset for binary fake news classification.
It also trains the baseline models (TF-IDF + LR/SVM) for direct comparison.

## ⚡ Before you start
1. **Runtime → Change runtime type → T4 GPU** (or A100 if available)
2. Run cells **top-to-bottom** in order
3. At the end, download `models/` to your local machine

---

### Expected results (LIAR binary, test set)
| Model | Accuracy | F1 (macro) |
|---|---|---|
| TF-IDF + LR | ~62% | ~61% |
| TF-IDF + SVM | ~64% | ~63% |
| **RoBERTa-base** | **~70–72%** | **~69–71%** |


## 1. Environment Setup

In [ ]:
# Verify GPU availability
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '⚠️  No GPU found. Please enable GPU runtime.')

In [ ]:
%%capture
!pip install transformers>=4.40 datasets>=2.19 accelerate>=0.29 \
             torch scikit-learn pandas numpy pyyaml \
             matplotlib seaborn

In [ ]:
import os
import sys
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Dataset — LIAR

In [ ]:
# Download the LIAR dataset
import urllib.request, zipfile

RAW_DIR = Path('data/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)

LIAR_URL = 'https://www.cs.ucsb.edu/~william/data/liar_dataset.zip'
EXPECTED = ['train.tsv', 'valid.tsv', 'test.tsv']

if all((RAW_DIR / f).exists() for f in EXPECTED):
    print('Dataset already downloaded.')
else:
    print('Downloading LIAR dataset...')
    zip_path = RAW_DIR / 'liar_dataset.zip'
    urllib.request.urlretrieve(LIAR_URL, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(RAW_DIR)
    zip_path.unlink()
    print('Done!')

for f in EXPECTED:
    size = (RAW_DIR / f).stat().st_size / 1024
    print(f'  {f}: {size:.0f} KB')

## 3. Preprocessing

In [ ]:
import re

LIAR_COLUMNS = [
    'id', 'label', 'statement', 'subject', 'speaker', 'speaker_job',
    'state_info', 'party', 'barely_true_counts', 'false_counts',
    'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context'
]

BINARY_LABEL_MAP = {
    'pants-fire': 0, 'false': 0, 'barely-true': 0,
    'half-true': 1,  'mostly-true': 1, 'true': 1
}

CREDIT_COLUMNS = ['barely_true_counts', 'false_counts', 'half_true_counts',
                  'mostly_true_counts', 'pants_on_fire_counts']

def clean_text(text):
    if not isinstance(text, str): return ''
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    return text.lower()

def preprocess(filepath, split_name):
    df = pd.read_csv(filepath, sep='\t', header=None, names=LIAR_COLUMNS,
                     dtype=str, na_values=[''])
    df = df.dropna(subset=['statement'])
    df = df[df['label'].isin(BINARY_LABEL_MAP)].reset_index(drop=True)
    for col in CREDIT_COLUMNS:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
    for col in ['subject','speaker','speaker_job','state_info','party','context']:
        df[col] = df[col].fillna('unknown').str.strip().str.lower().replace('', 'unknown')
    df['label_binary'] = df['label'].map(BINARY_LABEL_MAP)
    df['statement_clean'] = df['statement'].apply(clean_text)
    df['statement_len'] = df['statement_clean'].str.split().str.len()
    total = df[CREDIT_COLUMNS].sum(axis=1).replace(0, np.nan)
    df['speaker_lie_rate'] = ((df['false_counts'] + df['pants_on_fire_counts']) / total).fillna(0.0)
    print(f'{split_name}: {len(df)} rows | label dist: {dict(df["label_binary"].value_counts())}')
    return df

PROC_DIR = Path('data/processed')
PROC_DIR.mkdir(exist_ok=True)

train_df = preprocess(RAW_DIR / 'train.tsv', 'train')
valid_df = preprocess(RAW_DIR / 'valid.tsv', 'valid')
test_df  = preprocess(RAW_DIR / 'test.tsv',  'test')

train_df.to_csv(PROC_DIR / 'train_processed.csv', index=False)
valid_df.to_csv(PROC_DIR / 'valid_processed.csv', index=False)
test_df.to_csv(PROC_DIR  / 'test_processed.csv',  index=False)
print('Processed CSVs saved.')

## 4. Baseline Models (TF-IDF + LR/SVM)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, f1_score, accuracy_score
import pickle

TEXT_COL  = 'statement_clean'
LABEL_COL = 'label_binary'

# Combine train + valid for baseline (common practice)
trainval_df = pd.concat([train_df, valid_df], ignore_index=True)
X_tv = trainval_df[TEXT_COL].fillna('')
y_tv = trainval_df[LABEL_COL].astype(int)
X_test_bl = test_df[TEXT_COL].fillna('')
y_test_bl  = test_df[LABEL_COL].astype(int)

baseline_results = {}

# ── Logistic Regression ─────────────────────────────────────────────────────
print('Training TF-IDF + Logistic Regression (GridSearchCV)...')
lr_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf',   LogisticRegression()),
])
lr_grid = [{
    'tfidf__ngram_range': [(1,1),(1,2)],
    'tfidf__max_features': [10000, 50000],
    'tfidf__sublinear_tf': [True],
    'clf__C': [0.1, 1.0, 10.0],
    'clf__max_iter': [1000],
    'clf__solver': ['lbfgs'],
}]
lr_gs = GridSearchCV(lr_pipeline, lr_grid, cv=5, scoring='f1_macro', n_jobs=-1, verbose=1)
lr_gs.fit(X_tv, y_tv)
y_pred_lr = lr_gs.predict(X_test_bl)
print('\nBest params:', lr_gs.best_params_)
print(classification_report(y_test_bl, y_pred_lr, target_names=['Fake','Real']))
baseline_results['TF-IDF + LR'] = {
    'accuracy': accuracy_score(y_test_bl, y_pred_lr),
    'f1': f1_score(y_test_bl, y_pred_lr, average='macro', zero_division=0)
}

# ── LinearSVC ────────────────────────────────────────────────────────────────
print('\nTraining TF-IDF + LinearSVC (GridSearchCV)...')
svm_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf',   LinearSVC()),
])
svm_grid = [{
    'tfidf__ngram_range': [(1,1),(1,2)],
    'tfidf__max_features': [10000, 50000],
    'tfidf__sublinear_tf': [True],
    'clf__C': [0.01, 0.1, 1.0],
    'clf__max_iter': [2000],
}]
svm_gs = GridSearchCV(svm_pipeline, svm_grid, cv=5, scoring='f1_macro', n_jobs=-1, verbose=1)
svm_gs.fit(X_tv, y_tv)
y_pred_svm = svm_gs.predict(X_test_bl)
print('\nBest params:', svm_gs.best_params_)
print(classification_report(y_test_bl, y_pred_svm, target_names=['Fake','Real']))
baseline_results['TF-IDF + SVM'] = {
    'accuracy': accuracy_score(y_test_bl, y_pred_svm),
    'f1': f1_score(y_test_bl, y_pred_svm, average='macro', zero_division=0)
}

# Save
Path('models').mkdir(exist_ok=True)
with open('models/baseline_lr.pkl', 'wb') as f: pickle.dump(lr_gs.best_estimator_, f)
with open('models/baseline_svm.pkl', 'wb') as f: pickle.dump(svm_gs.best_estimator_, f)
print('\nBaseline models saved to models/')

## 5. RoBERTa Fine-tuning

In [ ]:
# ── Training Configuration ───────────────────────────────────────────────────
ROBERTA_CONFIG = {
    'model_name':       'roberta-base',
    'num_labels':       2,
    'max_length':       128,
    'dropout':          0.1,
    'batch_size':       16,
    'eval_batch_size':  32,
    'num_epochs':       4,
    'learning_rate':    2e-5,
    'warmup_ratio':     0.1,
    'weight_decay':     0.01,
    'fp16':             True,   # mixed precision (T4/A100)
    'seed':             42,
    'early_stopping_patience': 2,
    'metric_for_best_model': 'eval_f1',
}

print('Configuration:')
for k, v in ROBERTA_CONFIG.items():
    print(f'  {k}: {v}')

In [ ]:
import torch
from torch import nn
from torch.utils.data import Dataset
from transformers import RobertaTokenizer, RobertaModel

# ── Dataset ──────────────────────────────────────────────────────────────────
class FakeNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts    = list(texts)
        self.labels   = list(labels) if labels is not None else None
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx]) if self.texts[idx] is not None else ''
        enc  = self.tokenizer(
            text, truncation=True, padding='max_length',
            max_length=self.max_length, return_tensors='pt'
        )
        item = {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
        }
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# ── Model ─────────────────────────────────────────────────────────────────────
class RobertaClassifier(nn.Module):
    def __init__(self, model_name='roberta-base', num_labels=2, dropout=0.1):
        super().__init__()
        self.roberta    = RobertaModel.from_pretrained(model_name)
        self.dropout    = nn.Dropout(p=dropout)
        hidden_size     = self.roberta.config.hidden_size
        self.classifier = nn.Linear(hidden_size, num_labels)
        self.num_labels = num_labels

    def forward(self, input_ids, attention_mask, labels=None):
        out    = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(out.last_hidden_state[:, 0, :])
        logits = self.classifier(pooled)
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)
            return loss, logits
        return logits

print('Dataset and model classes defined.')

In [ ]:
from transformers import RobertaTokenizer

tokenizer = RobertaTokenizer.from_pretrained(ROBERTA_CONFIG['model_name'])

def make_dataset(df, text_col='statement_clean', label_col='label_binary'):
    df = df.dropna(subset=[text_col, label_col]).reset_index(drop=True)
    return FakeNewsDataset(
        df[text_col].astype(str).tolist(),
        df[label_col].astype(int).tolist(),
        tokenizer,
        ROBERTA_CONFIG['max_length']
    )

train_dataset = make_dataset(train_df)
valid_dataset = make_dataset(valid_df)
test_dataset  = make_dataset(test_df)

print(f'Datasets ready — train: {len(train_dataset)}, valid: {len(valid_dataset)}, test: {len(test_dataset)}')

In [ ]:
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.metrics import f1_score, accuracy_score

# Compute metrics callback
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': float(accuracy_score(labels, preds)),
        'f1':       float(f1_score(labels, preds, average='macro', zero_division=0)),
    }

# Initialise model
torch.manual_seed(ROBERTA_CONFIG['seed'])
model = RobertaClassifier(
    model_name=ROBERTA_CONFIG['model_name'],
    num_labels=ROBERTA_CONFIG['num_labels'],
    dropout=ROBERTA_CONFIG['dropout'],
)

# Training arguments
training_args = TrainingArguments(
    output_dir='models/roberta_checkpoints',
    num_train_epochs=ROBERTA_CONFIG['num_epochs'],
    per_device_train_batch_size=ROBERTA_CONFIG['batch_size'],
    per_device_eval_batch_size=ROBERTA_CONFIG['eval_batch_size'],
    learning_rate=ROBERTA_CONFIG['learning_rate'],
    warmup_ratio=ROBERTA_CONFIG['warmup_ratio'],
    weight_decay=ROBERTA_CONFIG['weight_decay'],
    fp16=ROBERTA_CONFIG['fp16'] and (DEVICE == 'cuda'),
    seed=ROBERTA_CONFIG['seed'],
    eval_strategy='steps',
    eval_steps=500,
    save_strategy='steps',
    save_steps=500,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model='eval_f1',
    greater_is_better=True,
    save_total_limit=2,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=ROBERTA_CONFIG['early_stopping_patience'])
    ],
)

print('Starting training...')
trainer.train()

In [ ]:
# Save best model
BEST_DIR = Path('models/roberta_best')
trainer.save_model(str(BEST_DIR))
tokenizer.save_pretrained(str(BEST_DIR))
print(f'Model saved to {BEST_DIR}')

# List saved files
for f in sorted(BEST_DIR.iterdir()):
    print(f'  {f.name} ({f.stat().st_size / 1e6:.1f} MB)')

## 6. Evaluation & Results

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print('=== RoBERTa Test Set Evaluation ===\n')
test_output = trainer.predict(test_dataset)
y_pred_roberta = np.argmax(test_output.predictions, axis=-1)
y_true         = test_df.dropna(subset=['statement_clean','label_binary']).reset_index(drop=True)['label_binary'].astype(int).values

print(classification_report(y_true, y_pred_roberta, target_names=['Fake','Real']))

roberta_metrics = {
    'accuracy': float(accuracy_score(y_true, y_pred_roberta)),
    'f1':       float(f1_score(y_true, y_pred_roberta, average='macro', zero_division=0))
}
print('RoBERTa metrics:', roberta_metrics)

In [ ]:
# Confusion matrices for all models
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models_preds = [
    ('TF-IDF + LR',  y_pred_lr),
    ('TF-IDF + SVM', y_pred_svm),
    ('RoBERTa',      y_pred_roberta),
]

for ax, (name, y_pred) in zip(axes, models_preds):
    cm = confusion_matrix(y_test_bl if 'roberta' not in name.lower() else y_true, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=['Fake','Real'], yticklabels=['Fake','Real'], ax=ax)
    ax.set_title(f'{name}\nAcc={accuracy_score(y_test_bl if "roberta" not in name.lower() else y_true, y_pred):.3f}  F1={f1_score(y_test_bl if "roberta" not in name.lower() else y_true, y_pred, average="macro"):.3f}',
                 fontsize=11)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')

plt.suptitle('Confusion Matrices — All Models (Row-Normalised)', fontsize=13, y=1.02)
plt.tight_layout()
Path('models/roberta_results').mkdir(parents=True, exist_ok=True)
plt.savefig('models/roberta_results/confusion_matrices_all.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

In [ ]:
# ── Model comparison table ───────────────────────────────────────────────────
comparison = pd.DataFrame({
    'Model':    ['TF-IDF + LR', 'TF-IDF + SVM', 'RoBERTa-base'],
    'Accuracy': [baseline_results['TF-IDF + LR']['accuracy'],
                 baseline_results['TF-IDF + SVM']['accuracy'],
                 roberta_metrics['accuracy']],
    'F1 (macro)': [baseline_results['TF-IDF + LR']['f1'],
                   baseline_results['TF-IDF + SVM']['f1'],
                   roberta_metrics['f1']],
})

print('=== Model Comparison — Test Set ===')
print(comparison.to_string(index=False, float_format='{:.4f}'.format))

# Bar chart comparison
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(comparison))
w = 0.35
ax.bar(x - w/2, comparison['Accuracy'], w, label='Accuracy', color='steelblue')
ax.bar(x + w/2, comparison['F1 (macro)'], w, label='F1 (macro)', color='coral')
ax.set_xticks(x); ax.set_xticklabels(comparison['Model'], fontsize=11)
ax.set_ylim(0.5, 0.85); ax.set_ylabel('Score')
ax.set_title('Model Comparison — Accuracy & F1 (macro)')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('models/roberta_results/model_comparison.png', dpi=150)
plt.show()

## 7. Error Analysis

In [ ]:
# Identify misclassified examples for RoBERTa
test_clean = test_df.dropna(subset=['statement_clean','label_binary']).reset_index(drop=True)
error_mask  = y_true != y_pred_roberta

error_df = test_clean[error_mask].copy()
error_df['true_label']      = y_true[error_mask]
error_df['predicted_label'] = y_pred_roberta[error_mask]

# Error type breakdown
# False Positives: model predicted REAL (1) but was FAKE (0)
fp_df = error_df[error_df['predicted_label'] == 1]  # false positives
fn_df = error_df[error_df['predicted_label'] == 0]  # false negatives

print(f'Total errors: {error_mask.sum()} / {len(y_true)} ({100*error_mask.mean():.1f}%)')
print(f'  False Positives (predicted REAL, was FAKE): {len(fp_df)}')
print(f'  False Negatives (predicted FAKE, was REAL): {len(fn_df)}')

print('\n--- False Positives (model too credulous) ---')
print(fp_df[['statement','label','party','speaker_lie_rate']].head(5).to_string())

print('\n--- False Negatives (model too skeptical) ---')
print(fn_df[['statement','label','party','speaker_lie_rate']].head(5).to_string())

error_df.to_csv('models/roberta_results/error_analysis_roberta.csv', index=False)
print('\nError analysis saved.')

In [ ]:
# Error pattern analysis — party breakdown
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (df_sub, title) in zip(axes, [
    (fp_df, 'False Positives\n(Fake→Predicted Real)'),
    (fn_df, 'False Negatives\n(Real→Predicted Fake)'),
]):
    party_counts = df_sub['party'].value_counts().head(8)
    ax.barh(party_counts.index, party_counts.values, color='salmon')
    ax.set_title(title, fontsize=11); ax.set_xlabel('Count')
    ax.invert_yaxis()

plt.suptitle('Error Analysis — Party Affiliation Breakdown', fontsize=12)
plt.tight_layout()
plt.savefig('models/roberta_results/error_party_breakdown.png', dpi=150)
plt.show()

# Statement length distribution of errors vs correct
fig, ax = plt.subplots(figsize=(8, 4))
test_clean['error'] = error_mask
test_clean.groupby('error')['statement_len'].plot.hist(
    alpha=0.6, bins=30, ax=ax, density=True
)
ax.legend(['Correct', 'Error'])
ax.set_xlabel('Statement length (words)')
ax.set_title('Statement Length: Correct vs. Misclassified')
plt.tight_layout()
plt.savefig('models/roberta_results/error_length_dist.png', dpi=150)
plt.show()

## 8. Download Results

In [ ]:
# Zip and download the best model + results
import shutil

# Create zip archive of the best model
shutil.make_archive('roberta_best', 'zip', '.', 'models/roberta_best')
print('Created roberta_best.zip')

# Create zip archive of all results figures + CSVs
shutil.make_archive('roberta_results', 'zip', '.', 'models/roberta_results')
print('Created roberta_results.zip')

# In Google Colab, download via:
try:
    from google.colab import files
    files.download('roberta_best.zip')
    files.download('roberta_results.zip')
    print('Download started!')
except ImportError:
    print('Not in Colab — files saved locally as roberta_best.zip and roberta_results.zip')

## 9. Inference Demo

Demonstrate the trained model on new statements.

In [ ]:
import torch.nn.functional as F

model.eval()
LABEL_MAP = {0: '❌ FAKE', 1: '✅ REAL'}

def predict_text(text, max_length=128):
    """Run inference on a single text string."""
    enc = tokenizer(
        text, truncation=True, padding='max_length',
        max_length=max_length, return_tensors='pt'
    )
    with torch.inference_mode():
        logits = model(
            input_ids=enc['input_ids'].to(DEVICE),
            attention_mask=enc['attention_mask'].to(DEVICE),
        )
    probs   = F.softmax(logits, dim=-1).cpu().numpy()[0]
    pred_id = int(probs.argmax())
    return {
        'label':      LABEL_MAP[pred_id],
        'confidence': f'{probs[pred_id]*100:.1f}%',
        'prob_fake':  f'{probs[0]*100:.1f}%',
        'prob_real':  f'{probs[1]*100:.1f}%',
    }

test_statements = [
    'The unemployment rate fell to 3.5% in January.',
    'Vaccines have been proven to cause autism in children.',
    'Congress passed a bipartisan infrastructure bill.',
    'Scientists confirmed that drinking bleach cures COVID-19.',
    'The president vetoed the healthcare bill yesterday.',
]

print('=== Inference Demo ===\n')
for stmt in test_statements:
    result = predict_text(stmt)
    print(f'Statement: {stmt}')
    print(f'  Prediction: {result["label"]}  (confidence: {result["confidence"]})')
    print(f'  P(fake)={result["prob_fake"]}   P(real)={result["prob_real"]}\n')